# Train-test Split and Cleaning
Splitting the combined file. Cleaning of combined dataset.

## 1. Imports & Setup

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy.spatial import cKDTree

## 2. Preliminary Cleaning Before Split

In [18]:
df = pd.read_csv("combined.csv")

# Drop columns that are leakage or not useful for prediction
df = df.drop(columns=["date", "origin_lat", "origin_lon", "dest_lat", "dest_lon"])

# Fill NaN values in weather columns with median
weather_feature_cols = [c for c in df.columns if (c.startswith("origin_") or c.startswith("dest_")) and df[c].dtype != object]
df[weather_feature_cols] = df[weather_feature_cols].fillna(df[weather_feature_cols].median())    

# Encode carrier code
le = LabelEncoder()
df["Carrier Code"] = le.fit_transform(df["Carrier Code"])
carrier_mapping = dict(zip(le.transform(le.classes_), le.classes_))
    
# Encode origin and destination with shared mapping
all_airports = pd.Series(pd.concat([df["origin_code"], df["destination_code"]]).unique())
le.fit(all_airports)
airport_mapping = dict(zip(le.transform(le.classes_), le.classes_))
df["origin_code"] = le.transform(df["origin_code"])
df["destination_code"] = le.transform(df["destination_code"])
    
print("Carrier mapping:", carrier_mapping)
print("Airport mapping:", airport_mapping)

Carrier mapping: {np.int64(0): 'AA', np.int64(1): 'DL', np.int64(2): 'UA', np.int64(3): 'WN'}
Airport mapping: {np.int64(0): 'ABQ', np.int64(1): 'AGS', np.int64(2): 'ALB', np.int64(3): 'AMA', np.int64(4): 'ANC', np.int64(5): 'ATL', np.int64(6): 'ATW', np.int64(7): 'AUS', np.int64(8): 'AVL', np.int64(9): 'AVP', np.int64(10): 'BDL', np.int64(11): 'BFL', np.int64(12): 'BGR', np.int64(13): 'BHM', np.int64(14): 'BIL', np.int64(15): 'BIS', np.int64(16): 'BLI', np.int64(17): 'BNA', np.int64(18): 'BOI', np.int64(19): 'BOS', np.int64(20): 'BQN', np.int64(21): 'BTR', np.int64(22): 'BTV', np.int64(23): 'BUF', np.int64(24): 'BUR', np.int64(25): 'BWI', np.int64(26): 'BZN', np.int64(27): 'CAE', np.int64(28): 'CHA', np.int64(29): 'CHO', np.int64(30): 'CHS', np.int64(31): 'CID', np.int64(32): 'CLE', np.int64(33): 'CLT', np.int64(34): 'CMH', np.int64(35): 'COS', np.int64(36): 'CRP', np.int64(37): 'CVG', np.int64(38): 'DAB', np.int64(39): 'DAL', np.int64(40): 'DAY', np.int64(41): 'DCA', np.int64(42): 'D

In [19]:
cols = ['Carrier Code', 'destination_code', 'origin_code', 'scheduled_hour', 'month']

In [20]:
def one_hot(df, cols):
    """
    df: pandas DataFrame
    cols: a list of columns to encode 
    return a DataFrame with one-hot encoding
    """
    for each in cols:
        dummies = pd.get_dummies(df[each], prefix=each, drop_first=True) # creates the one-hot encoding cols
        df = pd.concat([df, dummies], axis=1)

    return df

cleaned_df = one_hot(df, cols)

## 3. Train-Test Split

In [21]:
X = cleaned_df.drop(['Departure delay (Minutes)',
            'Scheduled elapsed time (Minutes)', 'Actual elapsed time (Minutes)',
             'Taxi-Out time (Minutes)', 'day_of_week',
             'weather_delayed', 
            ], axis=1)
y = cleaned_df['weather_delayed'] # 'weather_delayed' for yes/no flight is delayed; 'Departure delay (Minutes)' for predicting minutes

X = X.astype(np.float32)

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## 3. Cleaning combined data

In [23]:
def clean(df):
    # Drop any remaining NaN rows
    df = df.dropna()

X_train = clean(X_train)
X_test = clean(X_test)
y_train = clean(y_train)
y_test = clean(y_test)